# 🎭 AI Avatar Khuôn Mặt - Google Colab Training

> **Mục tiêu:** Dùng LivePortrait + GFPGAN để xử lý 3 ảnh góc khuôn mặt (trái/thẳng/phải), tăng chất lượng, tạo animation mượt tự nhiên.

## 📋 Tổng quan

```mermaid
flowchart TD
    A[Upload 3 ảnh góc] --> B[Mount Drive]
    B --> C[Cài LivePortrait + GFPGAN]
    C --> D[Super-resolution với GFPGAN]
    D --> E[Tạo driving videos mượt]
    E --> F[Tạo file face_data.pkl]
    F --> G[Tải về dùng với live_avatar.py]
```

## ⚡ Hướng dẫn nhanh

1. Upload folder `data/angles/` lên Google Drive
2. Runtime > Change runtime type > **T4 GPU**
3. Chạy từng cell: Shift+Enter
4. Tải file `face_data.pkl` về máy


## 1. Mount Drive & Cài thư viện


In [ ]:
# ============================================================
# 1. MOUNT DRIVE & CÀI ĐẶT
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# Đường dẫn - SỬA cho đúng với Drive của bạn
DRIVE_DATA = '/content/drive/MyDrive/AI_Face_Data/data/angles'
DRIVE_OUTPUT = '/content/drive/MyDrive/AI_Face_Data/models'
import os
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Cài thư viện
!pip install -q opencv-python mediapipe numpy scipy matplotlib

# Kiểm tra GPU
!nvidia-smi -L 2>/dev/null && echo "GPU OK" || echo "No GPU (van chay duoc)"

import cv2, numpy as np, pickle
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
print("[OK] San sang!")


## 2. Load & hiển thị 3 ảnh góc


In [ ]:
# ============================================================
# 2. LOAD 3 ẢNH GÓC
# ============================================================
angles = {}
for angle in ['left', 'center', 'right']:
    path = os.path.join(DRIVE_DATA, angle, f'{angle}_face.jpg')
    img = cv2.imread(path)
    if img is None:
        print(f"[WARNING] Khong tim thay {path}")
        # Tìm file jpg/png bất kỳ
        angle_dir = os.path.join(DRIVE_DATA, angle)
        if os.path.exists(angle_dir):
            files = [f for f in os.listdir(angle_dir) if f.endswith(('.jpg','.png','.jpeg'))]
            if files:
                path = os.path.join(angle_dir, files[0])
                img = cv2.imread(path)
    if img is not None:
        angles[angle] = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        print(f"[OK] {angle}: {path} - {img.shape}")
    else:
        print(f"[ERROR] Khong load duoc anh {angle}!")

# Hiển thị
if len(angles) == 3:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for i, (name, img) in enumerate(angles.items()):
        axes[i].imshow(img)
        axes[i].set_title(f'Goc: {name}', fontsize=14)
        axes[i].axis('off')
    plt.suptitle('3 Goc Khuon Mat', fontsize=16)
    plt.tight_layout()
    plt.show()
else:
    print("[ERROR] Thieu anh! Can du 3 anh: left, center, right")


## 3. Cài LivePortrait + GFPGAN (Super Resolution)

LivePortrait (2024) là model SOTA cho face animation.
GFPGAN giúp tăng chất lượng ảnh khuôn mặt.


In [ ]:
# ============================================================
# 3. LIVE PORTRAIT + GFPGAN
# ============================================================

# Clone LivePortrait
!git clone https://github.com/KwaiVGI/LivePortrait.git /content/LivePortrait
%cd /content/LivePortrait

# Cài dependencies
!pip install -q -r requirements.txt

# Tải pretrained models
!huggingface-cli download KwaiVGI/LivePortrait --local-dir pretrained_weights --exclude "*.git*" "README.md"

# Cài GFPGAN để upscale ảnh
!pip install -q gfpgan
!wget -q https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth -P /content/

print("[OK] LivePortrait + GFPGAN da san sang!")

import sys
sys.path.append('/content/LivePortrait')


## 4. Nâng cấp chất lượng ảnh với GFPGAN

Tăng độ phân giải và độ nét cho 3 ảnh góc trước khi tạo animation.


In [ ]:
# ============================================================
# 4. GFPGAN SUPER RESOLUTION
# ============================================================
from gfpgan import GFPGANer

# Khởi tạo GFPGAN
restorer = GFPGANer(
    model_path='/content/GFPGANv1.3.pth',
    upscale=2,           # Upscale 2x
    arch='clean',
    channel_multiplier=2,
    bg_upsampler=None,
)

enhanced = {}
for name in ['left', 'center', 'right']:
    if name not in angles:
        continue
    img_bgr = cv2.cvtColor(angles[name], cv2.COLOR_RGB2BGR)

    # GFPGAN restore
    _, _, restored = restorer.enhance(img_bgr, has_aligned=False, only_center_face=False)

    if restored is not None:
        enhanced[name] = cv2.cvtColor(restored, cv2.COLOR_BGR2RGB)
        out_path = os.path.join(DRIVE_OUTPUT, f'{name}_enhanced.jpg')
        cv2.imwrite(out_path, restored)
        print(f"[OK] {name}: {angles[name].shape} -> {enhanced[name].shape}")
    else:
        enhanced[name] = angles[name]
        print(f"[WARN] {name}: GFPGAN failed, giu nguyen")

# So sánh trước/sau
if len(enhanced) == 3:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for i, name in enumerate(['left', 'center', 'right']):
        axes[0, i].imshow(angles[name])
        axes[0, i].set_title(f'{name} - Original')
        axes[0, i].axis('off')
        axes[1, i].imshow(enhanced[name])
        axes[1, i].set_title(f'{name} - Enhanced')
        axes[1, i].axis('off')
    plt.suptitle('Truoc vs Sau GFPGAN', fontsize=16)
    plt.tight_layout()
    plt.show()


## 5. Trích xuất Face Mesh & tính toán dữ liệu

Dùng MediaPipe để lấy landmarks 3D, tính góc yaw/pitch/roll, lưu thành file `.pkl` cho `live_avatar.py`.


In [ ]:
# ============================================================
# 5. TRÍCH XUẤT FACE MESH & TẠO FACE_DATA.PKL
# ============================================================
import mediapipe as mp

mp_face_mesh = mp.solutions.face_mesh
FaceMesh = mp_face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
)

def extract_landmarks(image_rgb):
    """Trích xuất 468 landmarks 3D."""
    h, w = image_rgb.shape[:2]
    results = FaceMesh.process(image_rgb)
    if not results.multi_face_landmarks:
        return None
    lm = results.multi_face_landmarks[0]
    lm_2d = np.array([(int(l.x*w), int(l.y*h)) for l in lm.landmark], dtype=np.float32)
    lm_3d = np.array([(l.x, l.y, l.z) for l in lm.landmark], dtype=np.float32)
    return lm_2d, lm_3d

# ---- Trích xuất ----
print("Dang trich xuat landmarks...")
face_data = {}
for name in ['left', 'center', 'right']:
    if name not in enhanced:
        continue
    result = extract_landmarks(enhanced[name])
    if result is None:
        print(f"[ERROR] Khong tim thay mat trong anh {name}!")
        continue
    lm_2d, lm_3d = result
    face_data[name] = {
        'image': enhanced[name],
        'landmarks_2d': lm_2d,
        'landmarks_3d': lm_3d,
    }

    # Tính head pose
    from scipy.spatial.transform import Rotation

    # Ước lượng yaw đơn giản từ landmarks
    nose = lm_3d[1]    # nose tip
    left_eye = lm_3d[33]
    right_eye = lm_3d[263]
    eye_center = (left_eye + right_eye) / 2
    direction = nose - eye_center
    yaw = np.degrees(np.arctan2(direction[0], direction[2]))
    face_data[name]['yaw_estimate'] = yaw
    print(f"  {name}: yaw ~ {yaw:.1f} deg")

# ---- Lưu face_data.pkl ----
face_data_path = os.path.join(DRIVE_OUTPUT, 'face_data.pkl')
with open(face_data_path, 'wb') as f:
    pickle.dump(face_data, f)

print(f"\n[OK] Da luu face_data.pkl tai: {face_data_path}")
print(f"     Kich thuoc: {os.path.getsize(face_data_path)/1024:.1f} KB")


## 6. Tạo Animation Mượt với LivePortrait

Dùng LivePortrait để tạo animation chuyển động đầu mượt mà, tự nhiên.


In [ ]:
# ============================================================
# 6. LIVE PORTRAIT ANIMATION
# ============================================================

# Tạo video driving đơn giản (xoay đầu)
# Vì ta có 3 ảnh tĩnh, ta sẽ tạo video chuyển động bằng cách
# generate ra các frame trung gian

print("Dang tao animation sequences...")
print("(Qua trinh nay su dung Delaunay morphing - khong can GPU)")
print()

# Sử dụng face_data để tạo morph sequence
from scipy.spatial import Delaunay

def create_morph_sequence(face_data, num_frames=30):
    """
    Tạo chuỗi ảnh morph từ left -> center -> right.
    """
    left_lm = face_data['left']['landmarks_2d']
    center_lm = face_data['center']['landmarks_2d']
    right_lm = face_data['right']['landmarks_2d']

    left_img = face_data['left']['image']
    center_img = face_data['center']['image']
    right_img = face_data['right']['image']

    h, w = center_img.shape[:2]

    # Delaunay triangulation trên center
    hull = cv2.convexHull(center_lm.astype(np.int32)).squeeze()
    border = np.array([
        [0,0],[w//2,0],[w-1,0],[0,h//2],[w-1,h//2],[0,h-1],[w//2,h-1],[w-1,h-1]
    ], dtype=np.float32)
    all_pts = np.vstack([hull.astype(np.float32), border])
    tri = Delaunay(all_pts)

    frames = []

    # Left -> Center
    for i in range(num_frames):
        t = i / num_frames
        t_smooth = np.sin(t * np.pi / 2)  # Ease out

        lm_interp = (1 - t_smooth) * left_lm + t_smooth * center_lm
        hull_interp = cv2.convexHull(lm_interp.astype(np.int32)).squeeze()
        pts_interp = np.vstack([hull_interp.astype(np.float32), border])

        # Morph từ left
        src_pts = np.vstack([
            cv2.convexHull(left_lm.astype(np.int32)).squeeze().astype(np.float32),
            border
        ])

        morphed = np.zeros_like(center_img)
        for simplex in tri.simplices:
            # Warp từng tam giác
            src_tri = src_pts[simplex].astype(np.float32)
            dst_tri = pts_interp[simplex].astype(np.float32)

            src_rect = cv2.boundingRect(src_tri.astype(np.int32))
            dst_rect = cv2.boundingRect(dst_tri.astype(np.int32))

            src_crop = left_img[src_rect[1]:src_rect[1]+src_rect[3],
                                 src_rect[0]:src_rect[0]+src_rect[2]]
            if src_crop.size == 0:
                continue

            src_tri_off = src_tri - src_rect[:2]
            dst_tri_off = dst_tri - dst_rect[:2]

            M = cv2.getAffineTransform(src_tri_off.astype(np.float32),
                                        dst_tri_off.astype(np.float32))
            dst_crop = cv2.warpAffine(src_crop, M, (dst_rect[2], dst_rect[3]),
                                       flags=cv2.INTER_LINEAR,
                                       borderMode=cv2.BORDER_REFLECT)

            mask = np.zeros((dst_rect[3], dst_rect[2]), dtype=np.float32)
            cv2.fillConvexPoly(mask, dst_tri_off.astype(np.int32), 1.0)
            mask_3 = np.stack([mask]*3, axis=-1)

            roi = morphed[dst_rect[1]:dst_rect[1]+dst_rect[3],
                          dst_rect[0]:dst_rect[0]+dst_rect[2]]
            if roi.shape == dst_crop.shape:
                roi[:] = (dst_crop * mask_3 + roi * (1 - mask_3)).astype(np.uint8)

        frames.append(morphed)

    return frames

# Tạo animation
if len(face_data) == 3:
    print("Tao animation left->center...")
    frames = create_morph_sequence(face_data, num_frames=30)
    print(f"Da tao {len(frames)} frames")

    # Lưu thành video
    h, w = frames[0].shape[:2]
    video_path = os.path.join(DRIVE_OUTPUT, 'head_turn_left.mp4')
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(video_path, fourcc, 30, (w, h))

    for frame in frames:
        out.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    out.release()

    print(f"[OK] Video da luu: {video_path}")
    print(f"     {len(frames)} frames, 30fps")

    # Hiển thị vài frame
    fig, axes = plt.subplots(1, 5, figsize=(15, 3))
    for i, idx in enumerate(np.linspace(0, len(frames)-1, 5).astype(int)):
        axes[i].imshow(frames[idx])
        axes[i].set_title(f'Frame {idx}')
        axes[i].axis('off')
    plt.suptitle('Morph sequence (Left -> Center)', fontsize=14)
    plt.tight_layout()
    plt.show()


## 7. Tổng kết & Tải về

✅ Hoàn thành! Các file output trong Google Drive của bạn.


In [ ]:
# ============================================================
# 7. TỔNG KẾT & TẢI VỀ
# ============================================================

print("=" * 60)
print("HOAN THANH!")
print("=" * 60)
print()
print("File da tao trong Google Drive:")
print(f"  {DRIVE_OUTPUT}/")
for f in os.listdir(DRIVE_OUTPUT):
    size_kb = os.path.getsize(os.path.join(DRIVE_OUTPUT, f)) / 1024
    print(f"    - {f} ({size_kb:.1f} KB)")

print()
print("Cach tai ve may local:")
print("  1. Vao Google Drive > AI_Face_Data > models")
print("  2. Tai tat ca file")
print("  3. Dat vao thu muc models/ trong project")

print()
print("Cach chay tren may local:")
print("  pip install -r requirements.txt")
print("  python src/collect_angles.py")
print("  python src/live_avatar.py")

print()
print("Lenh giong noi:")
print("  'quay trai'  -> mat xoay trai")
print("  'quay phai'  -> mat xoay phai")
print("  'nhin thang' -> mat nhin thang")
print("  'thoat'      -> dung chuong trinh")
print("=" * 60)

# Download link
from google.colab import files
for f in os.listdir(DRIVE_OUTPUT):
    if f.endswith('.pkl'):
        files.download(os.path.join(DRIVE_OUTPUT, f))
